# 05 · PagedAttention 与前缀缓存

第 04 章留了一个坑：`ModelRunner` 为了把不同进度的序列拼进一个 batch，必须把 KV **右填充到同一长度**。一个 batch 里混进一条长序列，所有序列都要按最长的那条分配显存。

这一章补上 vLLM 的 KV 管理层，同时解决另一个问题：**大量请求共享同一段 prompt 前缀，却各自重算一遍。**

## 架构位置

第 04 章说过调度和执行是分开的，现在补上第三块：

```
EngineCore.step()
  ├─ Scheduler.schedule()                   决定跑什么        ← 第 04 章
  │    └─ KVCacheManager.allocate_slots()   决定 KV 放哪       ← 本章
  ├─ ModelRunner.execute_model()            跑模型
  └─ Scheduler.update_from_output()
```

本章的类对应关系：

| 本章 | vLLM 源码 |
|---|---|
| `KVCacheBlock` | `vllm/v1/core/kv_cache_utils.py` |
| `BlockPool` | `vllm/v1/core/block_pool.py` |
| `KVCacheManager` | `vllm/v1/core/kv_cache_manager.py` |
| `hash_block_tokens()` | `vllm/v1/core/kv_cache_utils.py` |

In [ ]:
# ===== 引导单元：环境检查 + 测量工具 + MiniGPT（每章自带，直接运行）=====
# 说明：本单元在每个 notebook 里都有一份完整副本，目的是让任何一个 notebook
#       都能在 Colab 里零配置独立运行。想改模型结构，请改 tools/build_notebooks.py
#       里的 SETUP_CODE，然后重跑编译脚本。
#
# 架构对齐：下面这套推理核心刻意模仿了 vLLM V1 的模块划分与命名，
#   详见 docs/vllm-mapping.md 的对照表。
#       EngineCore.step()           ←→ vllm/v1/engine/core.py
#         ├─ Scheduler.schedule()   ←→ vllm/v1/core/sched/scheduler.py
#         ├─ ModelRunner.execute_model() ←→ vllm/v1/worker/gpu_model_runner.py
#         └─ Scheduler.update_from_output()
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# MiniGPT 只有 2700 万参数，用 float16 跑在 GPU 上；CPU 上 float16 很慢，用 float32
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# 常见卡的关键参数（近似值）。如果你的卡不在表里，直接在这里补一行：
#   "你的卡型号": {"mem_gb": .., "bw_gbps": .., "fp16_tflops": .., "arch": ".."},
# 三个数字都能在厂商 datasheet 上查到。第 02、03 章会用到它们。
CARD_SPECS = {
    "Tesla T4":        {"mem_gb": 16, "bw_gbps": 320,  "fp16_tflops": 65,  "arch": "Turing sm75"},
    "Tesla V100":      {"mem_gb": 16, "bw_gbps": 900,  "fp16_tflops": 125, "arch": "Volta sm70"},
    "A100-SXM4-40GB":  {"mem_gb": 40, "bw_gbps": 1555, "fp16_tflops": 312, "arch": "Ampere sm80"},
    "A100-SXM4-80GB":  {"mem_gb": 80, "bw_gbps": 2039, "fp16_tflops": 312, "arch": "Ampere sm80"},
    "L4":              {"mem_gb": 24, "bw_gbps": 300,  "fp16_tflops": 121, "arch": "Ada sm89"},
    "A10G":            {"mem_gb": 24, "bw_gbps": 600,  "fp16_tflops": 125, "arch": "Ampere sm86"},
    "H100 PCIe":       {"mem_gb": 80, "bw_gbps": 2000, "fp16_tflops": 756, "arch": "Hopper sm90"},
    "H100 80GB HBM3":  {"mem_gb": 80, "bw_gbps": 3350, "fp16_tflops": 989, "arch": "Hopper sm90"},
}


def lookup_card():
    """按 GPU 名称匹配规格表。匹配不到就返回零值，提醒你手工补。"""
    if not torch.cuda.is_available():
        return {"name": "CPU", "mem_gb": 0, "bw_gbps": 0, "fp16_tflops": 0, "arch": "CPU"}
    name = torch.cuda.get_device_properties(0).name
    for key, spec in CARD_SPECS.items():
        # 双向包含匹配：Colab 可能报 "Tesla T4"，也可能报 "NVIDIA L4"
        if key.lower() in name.lower() or name.lower().replace("nvidia ", "") in key.lower():
            return {"name": name, **spec}
    return {
        "name": name,
        "mem_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 1),
        "bw_gbps": 0,
        "fp16_tflops": 0,
        "arch": "未知卡型 → 请查 datasheet 后补进 CARD_SPECS",
    }


SPEC = lookup_card()


def sync():
    """GPU 是异步执行的，计时前必须同步，否则测到的是下发时间不是执行时间。"""
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def bench(fn, warmup=3, iters=10):
    """返回单次调用的平均耗时（毫秒）。warmup 用来排除首次 kernel 编译等开销。"""
    for _ in range(warmup):
        fn()
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    sync()
    return (time.perf_counter() - t0) / iters * 1000.0


def peak_mem_mb():
    """当前 CUDA 峰值显存占用（MB）。"""
    if DEVICE != "cuda":
        return 0.0
    return torch.cuda.max_memory_allocated() / 1024 ** 2


def reset_peak():
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()


class Config:
    def __init__(self, vocab_size=50257, block_size=1024, n_layer=4, n_head=6, n_embd=384):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head


class CausalSelfAttention(nn.Module):
    """因果自注意力，支持 KV cache。

    past_kv 传入历史的 (k, v)，本步只为新 token 计算 Q/K/V，然后拼在历史后面。
    返回 (输出, 更新后的 (k, v))，其中 k/v 的 shape 是 (B, n_head, 总长度, head_dim)。
    """

    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.head_dim
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x, past_kv=None, attn_mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)

        S = k.size(2)  # 总长度 = 历史 + 本步新增
        if attn_mask is None:
            # 默认因果掩码：本步第 i 个 query 的绝对位置是 S-T+i，只能看见 <= 它的 key
            mask = torch.ones(T, S, device=x.device).tril(diagonal=S - T).bool()
        else:
            # 外部传入的掩码，用于一个 batch 里混合不同进度的序列（第 04、06 章）
            mask = attn_mask
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), (k, v)


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x, past_kv=None, attn_mask=None):
        h, present = self.attn(self.ln_1(x), past_kv, attn_mask)
        x = x + h
        x = x + self.mlp(self.ln_2(x))
        return x, present


class MiniGPT(nn.Module):
    """极简 GPT，结构与 Llama 同源：pre-norm + 因果注意力 + 4 倍扩张 MLP + 权重共享。

    与 Llama 的两处差异：
      - 用可学习位置编码代替 RoPE（简化实现，不影响调度实验的结论）
      - 没有 GQA（本仓库是 MHA，第 03 章会手工比较两者的 KV cache 大小）
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # 权重共享，省一份 embedding 参数

        def init(m):
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

        self.apply(init)

    def forward(self, idx, past_kvs=None, pos_offset=0, attn_mask=None):
        """idx: (B, T) 的 token id。

        past_kvs: 长度等于层数的列表，每项是 (k, v)；None 表示从零开始（prefill）。
        pos_offset: 本次输入的第一个 token 的绝对位置。传 int 表示整个 batch 用同一个
                    偏移；传 shape (B,) 的张量表示每条序列各用各的偏移——当 batch 里
                    混合了不同进度的请求时必须这样传。
        attn_mask: 可选的自定义注意力掩码，用于屏蔽填充位。
        """
        B, T = idx.shape
        if torch.is_tensor(pos_offset):
            pos = pos_offset.view(B, 1) + torch.arange(T, device=idx.device)[None, :]
        else:
            pos = torch.arange(pos_offset, pos_offset + T, device=idx.device)[None, :].expand(B, T)
        x = self.wte(idx) + self.wpe(pos)

        presents = []
        for i, blk in enumerate(self.blocks):
            past = None if past_kvs is None else past_kvs[i]
            x, present = blk(x, past, attn_mask)
            presents.append(present)
        return self.lm_head(self.ln_f(x)), presents

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def build_model(seed=0, device=DEVICE, dtype=DTYPE, **kw):
    torch.manual_seed(seed)
    cfg = Config(**kw)
    model = MiniGPT(cfg).to(device=device, dtype=dtype)
    return model.eval()


@torch.no_grad()
def generate_naive(model, idx, max_new_tokens):
    """不用 KV cache：每一步都把完整序列重新算一遍（O(n^2) 重算）。"""
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -model.cfg.block_size:])
        idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return idx


@torch.no_grad()
def generate_cached(model, idx, max_new_tokens):
    """用 KV cache：prompt 只 prefill 一次，之后每步只喂 1 个 token。"""
    logits, past = model(idx)
    nxt = logits[:, -1].argmax(-1, keepdim=True)
    out = [nxt]
    pos = idx.size(1)
    for _ in range(max_new_tokens - 1):
        logits, past = model(nxt, past_kvs=past, pos_offset=pos)
        pos += 1
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        out.append(nxt)
    return torch.cat([idx] + out, dim=1)


def kv_bytes(n_layer, n_kv_head, head_dim, seq_len, batch=1, dtype_bytes=2):
    """KV cache 字节数。注意是 2（K 和 V 各一份）。"""
    return 2 * n_layer * n_kv_head * head_dim * seq_len * batch * dtype_bytes


# ========== 以下是模仿 vLLM V1 架构的推理核心 ==========


class Request:
    """对应 vllm/v1/request.py 的 Request。

    num_computed_tokens 是 vLLM 里最核心的一个字段：它记录这条请求已经有
    多少 token 的 KV 被算过。prefill、chunked prefill、前缀缓存命中——
    三种看起来完全不同的场景，在 vLLM 里都只是「把 num_computed_tokens 往前推」。
    理解这一点，chunked prefill 就不再是独立机制，而是这个字段的自然结果。
    """

    def __init__(self, request_id, prompt_token_ids, max_tokens):
        self.request_id = request_id
        self.prompt_token_ids = list(prompt_token_ids)
        self.max_tokens = max_tokens
        self.output_token_ids = []
        self.num_computed_tokens = 0
        self.status = "waiting"      # waiting / running / finished
        # 本仓库简化：直接把 KV 张量挂在请求上。
        # 真实 vLLM 不这么做——请求只持有 block_table，物理 block 由 KVCacheManager 管（第 05 章）。
        self.past = None

    @property
    def num_prompt_tokens(self):
        return len(self.prompt_token_ids)

    def all_token_ids(self):
        return self.prompt_token_ids + self.output_token_ids

    def num_tokens_to_schedule(self):
        """还欠多少 token 没算：prefill 阶段是剩余 prompt 长度，decode 阶段是 1。"""
        if self.num_computed_tokens < self.num_prompt_tokens:
            return self.num_prompt_tokens - self.num_computed_tokens
        return 1

    @property
    def is_finished(self):
        return len(self.output_token_ids) >= self.max_tokens

    def __repr__(self):
        return (f"Request({self.request_id}, computed={self.num_computed_tokens}"
                f"/{self.num_prompt_tokens}, out={len(self.output_token_ids)}"
                f"/{self.max_tokens}, {self.status})")


class SchedulerOutput:
    """对应 vllm/v1/core/sched/output.py 的 SchedulerOutput。

    调度与执行之间唯一的接口。真实 vLLM 里这个结构还包含 block 分配结果、
    抢占列表等字段，这里只保留最必要的两个。
    """

    def __init__(self, scheduled_reqs, num_scheduled_tokens):
        self.scheduled_reqs = scheduled_reqs
        self.num_scheduled_tokens = num_scheduled_tokens   # {request_id: n}

    def __len__(self):
        return len(self.scheduled_reqs)


class Scheduler:
    """对应 vllm/v1/core/sched/scheduler.py 的 Scheduler。

    职责边界是这个架构里最值得学的一点：Scheduler 只决定
    「这一轮跑哪些请求、各自跑几个 token」，它既不碰显存也不碰模型。

        显存分配 → KVCacheManager（第 05 章）
        真正计算 → ModelRunner

    三个模块分离，才能各自独立替换实现。面试被问「说说 vLLM 的架构」时，
    先把这个职责划分讲清楚，比背模块名有用得多。
    """

    def __init__(self, max_num_seqs=8, max_num_batched_tokens=2048):
        self.waiting = []
        self.running = []
        self.finished = []
        self.max_num_seqs = max_num_seqs
        # 这个预算就是 chunked prefill 的开关：调小它，长 prompt 自然被切成多轮（第 06 章）
        self.max_num_batched_tokens = max_num_batched_tokens
        self.step_id = 0

    def add_request(self, req):
        self.waiting.append(req)

    def has_unfinished(self):
        return bool(self.waiting or self.running)

    def schedule(self):
        scheduled, num_tokens = [], {}
        budget = self.max_num_batched_tokens

        # 第一优先：正在跑的请求。已进 decode 的排 1 个 token；
        # 还在做 chunked prefill 的按剩余量排，但受 budget 限制。
        for req in list(self.running):
            if budget <= 0 or len(scheduled) >= self.max_num_seqs:
                break
            n = min(req.num_tokens_to_schedule(), budget)
            scheduled.append(req)
            num_tokens[req.request_id] = n
            budget -= n

        # 第二优先：从队列里补新请求进来做 prefill
        for req in list(self.waiting):
            if budget <= 0 or len(scheduled) >= self.max_num_seqs:
                break
            n = min(req.num_tokens_to_schedule(), budget)
            scheduled.append(req)
            num_tokens[req.request_id] = n
            budget -= n
            self.waiting.remove(req)
            req.status = "running"
            self.running.append(req)

        self.step_id += 1
        return SchedulerOutput(scheduled, num_tokens)

    def update_from_output(self, sched_out, sampled):
        """对应 vLLM 的 update_from_output：写回采样结果，处理完成与回收。

        本轮被调度但没产生 token 的请求（比如 chunked prefill 的中间块）
        不会出现在 sampled 里，它们保持 running，下一轮继续。
        """
        for req in sched_out.scheduled_reqs:
            if req.request_id not in sampled:
                continue
            req.output_token_ids.append(sampled[req.request_id])
            if req.is_finished:
                req.status = "finished"
                if req in self.running:
                    self.running.remove(req)
                self.finished.append(req)
                req.past = None      # 简化回收；真实 vLLM 走 KVCacheManager.free()


class ModelRunner:
    """对应 vllm/v1/worker/gpu_model_runner.py 的 GPUModelRunner。

    职责：把 Scheduler 排好的一批请求拼成一次前向，返回新采样的 token。

    与真实 vLLM 的差距（要如实知道）：
      · vLLM 用 block_table 让每条序列的 KV 物理上不连续，所以不需要填充；
        这里用「右填充 + 逐序列掩码」对齐，会浪费显存——第 05 章解决。
      · vLLM 会把 prefill 和 decode 混在同一个 batch 里跑；这里分成两组处理，
        纯粹是为了让代码可读，结论不受影响。
      · 输入准备、CUDA graph、attention metadata 这些都被省掉了。
    """

    def __init__(self, model):
        self.model = model

    @torch.no_grad()
    def _run_decode_batch(self, reqs):
        """把一批进度不同的 decode 请求拼成一次前向。"""
        B = len(reqs)
        lens = [r.num_computed_tokens for r in reqs]
        Lmax = max(lens)
        n_layer = self.model.cfg.n_layer

        padded = []
        for layer in range(n_layer):
            ks, vs = [], []
            for r in reqs:
                k, v = r.past[layer]
                pad = Lmax - k.size(2)
                if pad:
                    k = F.pad(k, (0, 0, 0, pad))
                    v = F.pad(v, (0, 0, 0, pad))
                ks.append(k)
                vs.append(v)
            padded.append((torch.cat(ks, 0), torch.cat(vs, 0)))

        # 逐序列掩码：真实历史 [0, L_i) + 新 token 落在下标 Lmax
        S = Lmax + 1
        mask = torch.zeros(B, 1, 1, S, dtype=torch.bool, device=DEVICE)
        for i, r in enumerate(reqs):
            mask[i, 0, 0, : lens[i]] = True
            mask[i, 0, 0, Lmax] = True

        ids = torch.tensor([[r.all_token_ids()[r.num_computed_tokens]] for r in reqs],
                           device=DEVICE)
        pos = torch.tensor(lens, device=DEVICE)
        logits, past = self.model(ids, past_kvs=padded, pos_offset=pos, attn_mask=mask)

        sampled = {}
        for i, r in enumerate(reqs):
            rebuilt = []
            for layer in range(n_layer):
                k_all, v_all = past[layer]
                k = torch.cat([k_all[i:i + 1, :, : lens[i]],
                               k_all[i:i + 1, :, Lmax:Lmax + 1]], dim=2)
                v = torch.cat([v_all[i:i + 1, :, : lens[i]],
                               v_all[i:i + 1, :, Lmax:Lmax + 1]], dim=2)
                rebuilt.append((k, v))
            r.past = rebuilt
            r.num_computed_tokens += 1
            sampled[r.request_id] = int(logits[i, -1].argmax(-1).item())
        return sampled

    @torch.no_grad()
    def execute_model(self, sched_out):
        decode_reqs, prefill_reqs = [], []
        for r in sched_out.scheduled_reqs:
            # 判断依据是「prompt 算完了没有」，而不是「本轮排了几个 token」
            if r.num_computed_tokens >= r.num_prompt_tokens:
                decode_reqs.append(r)
            else:
                prefill_reqs.append(r)

        sampled = {}
        if decode_reqs:
            sampled.update(self._run_decode_batch(decode_reqs))

        for r in prefill_reqs:
            n = sched_out.num_scheduled_tokens[r.request_id]
            start = r.num_computed_tokens
            chunk = r.all_token_ids()[start:start + n]
            toks = torch.tensor([chunk], device=DEVICE)
            logits, past = self.model(toks, past_kvs=r.past, pos_offset=start)
            r.past = past
            r.num_computed_tokens += len(chunk)
            # 只有 prompt 全部算完，才能采样第一个输出 token
            if r.num_computed_tokens >= r.num_prompt_tokens:
                sampled[r.request_id] = int(logits[:, -1].argmax(-1).item())
        return sampled


class EngineCore:
    """对应 vllm/v1/engine/core.py 的 EngineCore。

    整个 vLLM 的推理服务就跑在这三步上：

        schedule()            决定这一轮跑什么
        execute_model()       跑模型
        update_from_output()  把结果写回请求状态

    读懂这个循环你就抓住了 vLLM 的主干。后面所有优化——chunked prefill、
    前缀缓存、抢占、投机解码——都是在这三步里插桩。
    """

    def __init__(self, model, scheduler=None):
        self.scheduler = scheduler or Scheduler()
        self.runner = ModelRunner(model)
        self.step_id = 0
        self.steps = 0

    def step(self):
        sched_out = self.scheduler.schedule()
        if len(sched_out) == 0:
            return None
        sampled = self.runner.execute_model(sched_out)
        self.scheduler.update_from_output(sched_out, sampled)
        self.step_id += 1
        self.steps += 1
        return sampled

    def run(self, max_steps=10000):
        while self.scheduler.has_unfinished() and self.steps < max_steps:
            self.step()
        return self.steps


print(f"引导单元加载完成 | device={DEVICE} dtype={DTYPE} torch={torch.__version__}")
# ===== 引导单元结束 =====

In [ ]:
from collections import deque

print("本章只用引导单元里的 MiniGPT，调度部分用不到——因为 KV 管理层和调度层是解耦的。")

## 一、核心思路：把 KV cache 当虚拟内存管

操作系统怎么解决"进程需要连续内存，但物理内存会碎片"？**分页**——进程看到的是连续虚拟地址，实际映射到任意物理页，靠页表翻译。

PagedAttention 是同一个思路：

| 操作系统 | PagedAttention |
|---|---|
| 物理页 | KV block（固定大小，vLLM 默认 16 个 token） |
| 页表 | `block_table`（这条序列用了哪些 block） |
| 进程 | 一条请求序列 |
| 共享库（多进程共享代码页） | **前缀缓存**（多条请求共享同一批 block） |

关键收益：序列的 KV 不再需要物理连续，也**不需要为了对齐而填充**。按需分配，用多少给多少。

## 二、KVCacheBlock 与链式哈希

In [ ]:
class KVCacheBlock:
    """对应 vllm/v1/core/kv_cache_utils.py 的 KVCacheBlock。

    一个物理 block 只需要这几个字段：
      · ref_cnt    —— 被几条序列引用（前缀共享靠它）
      · block_hash —— 链式哈希值，None 表示还没填满、不可复用
    """

    __slots__ = ("block_id", "ref_cnt", "block_hash")

    def __init__(self, block_id):
        self.block_id = block_id
        self.ref_cnt = 0
        self.block_hash = None

    def __repr__(self):
        h = "----" if self.block_hash is None else f"{self.block_hash % 10000:04d}"
        return f"Block#{self.block_id}(ref={self.ref_cnt},hash={h})"


def hash_block_tokens(parent_block_hash, curr_block_token_ids):
    """对应 vllm/v1/core/kv_cache_utils.py 的 hash_block_tokens。

    链式哈希：哈希值 = H(前一个 block 的哈希, 当前 block 的 token)。

    为什么必须把 parent 算进去？如果只哈希当前 block 的内容，那么**相同内容出现在
    不同位置**时会被误判成可复用——但它们前面的上下文不同，KV 完全不同，
    一旦复用就会算错。这是链式哈希存在的唯一理由。
    """
    return hash((parent_block_hash, tuple(curr_block_token_ids)))


same = [1, 2, 3, 4]
h0 = hash_block_tokens(-1, same)
h1 = hash_block_tokens(h0, same)
print("链式哈希演示：")
print(f"  第一次出现的 hash : {h0 % 10000:04d}")
print(f"  换个位置再出现    : {h1 % 10000:04d}   ← 不同，所以不会被误复用")

## 三、BlockPool：物理 block 的分配与回收

In [ ]:
class BlockPool:
    """对应 vllm/v1/core/block_pool.py 的 BlockPool。

    管理全部物理 block：分配、回收、以及前缀哈希表。

    与真实 vLLM 的差异：vLLM 的 free blocks 用 FreeKVCacheBlockQueue（双向链表，
    O(1) 增删），并把"有哈希但没人引用"的 block 单独放进淘汰队列，
    让前缀缓存和回收可以同时成立。这里用 deque + 覆写时摘除哈希简化实现，
    语义等价，在讲解场景下行为一致。
    """

    def __init__(self, num_blocks, block_size):
        self.block_size = block_size
        self.blocks = [KVCacheBlock(i) for i in range(num_blocks)]
        self.free_blocks = deque(b.block_id for b in self.blocks)
        # 前缀哈希 → block，这就是 prefix cache 的本体
        self.cached_block_hash_to_block = {}

    def get_new_blocks(self, n):
        """对应 vLLM 的 get_new_blocks。"""
        if n > len(self.free_blocks):
            raise MemoryError(f"block 不足：需要 {n}，剩余 {len(self.free_blocks)}")
        ids = [self.free_blocks.popleft() for _ in range(n)]
        for i in ids:
            b = self.blocks[i]
            if b.block_hash is not None:
                # 这个 block 原来缓存着别的前缀，现在要被覆写，从缓存表里摘掉
                self.cached_block_hash_to_block.pop(b.block_hash, None)
                b.block_hash = None
            b.ref_cnt += 1
        return ids

    def free_blocks_of(self, block_ids):
        """对应 vLLM 的 free_blocks：引用计数减到 0 才真正回到空闲队列。"""
        for i in block_ids:
            b = self.blocks[i]
            b.ref_cnt -= 1
            if b.ref_cnt == 0:
                self.free_blocks.append(i)

    def cache_full_block(self, block_id, block_hash):
        """一个 block 被填满后注册进前缀哈希表，后续请求才可能命中。"""
        b = self.blocks[block_id]
        b.block_hash = block_hash
        self.cached_block_hash_to_block[block_hash] = b

    @property
    def num_free(self):
        return len(self.free_blocks)


pool = BlockPool(num_blocks=32, block_size=16)
print(f"启动一个 {len(pool.blocks)} 个 block、每块 16 token 的 KV 池"
      f"（总容量 {len(pool.blocks) * 16} token）")

## 四、KVCacheManager：前缀命中就是把 `num_computed_tokens` 推上去

**这是本章最重要的一段。** 第 04 章反复强调 `num_computed_tokens` 是理解 vLLM 的钥匙，现在你会看到它怎么和前缀缓存配合：

请求进来时先查前缀缓存。命中 N 个 block，就把 `num_computed_tokens` 直接设成 `N × block_size`——**这些 token 一个都不用算**，剩下的部分才进入正常的 prefill。

In [ ]:
class KVCacheManager:
    """对应 vllm/v1/core/kv_cache_manager.py 的 KVCacheManager。

    注意职责边界：它不决定"跑什么"（那是 Scheduler 的事），
    只回答两个问题——KV 放哪里、能复用多少。
    """

    def __init__(self, block_pool):
        self.block_pool = block_pool
        self.block_tables = {}          # request_id -> [block_id, ...]

    def _match_prefix_blocks(self, req):
        """逐块做链式哈希比对，返回从开头起能连续复用的 block 对象列表。

        注意末尾不满一个 block 的 token 永远匹配不上——因为它还会继续增长，
        这一块的内容还会变。这就是"最后一块不可复用"的根本原因。
        """
        bs = self.block_pool.block_size
        tokens = req.all_token_ids()
        parent_hash, matched = -1, []
        for start in range(0, len(tokens) // bs * bs, bs):
            h = hash_block_tokens(parent_hash, tokens[start:start + bs])
            blk = self.block_pool.cached_block_hash_to_block.get(h)
            if blk is None:
                break
            matched.append(blk)
            parent_hash = h
        return matched

    def attach_prefix_cache(self, req):
        """前缀缓存命中时的动作：零拷贝共享 block + 推进 num_computed_tokens。"""
        matched = self._match_prefix_blocks(req)
        if not matched:
            return 0
        for blk in matched:
            blk.ref_cnt += 1                       # 共享，不是复制
        self.block_tables[req.request_id] = [b.block_id for b in matched]
        req.num_computed_tokens = len(matched) * self.block_pool.block_size
        return len(matched)

    def allocate_slots(self, req, num_tokens):
        """为本轮要计算的 num_tokens 个 token 补齐所需 block。"""
        bs = self.block_pool.block_size
        need = math.ceil((req.num_computed_tokens + num_tokens) / bs)
        table = self.block_tables.setdefault(req.request_id, [])
        if need > len(table):
            table.extend(self.block_pool.get_new_blocks(need - len(table)))
        return table

    def cache_blocks(self, req):
        """把已经填满的 block 注册进前缀缓存，供后续请求复用（对应 cache_full_blocks）。"""
        bs = self.block_pool.block_size
        tokens = req.all_token_ids()
        table = self.block_tables.get(req.request_id, [])
        parent_hash, n_cached = -1, 0
        for idx, block_id in enumerate(table):
            start = idx * bs
            if start + bs > req.num_computed_tokens:
                break                      # 还没填满，不能缓存
            h = hash_block_tokens(parent_hash, tokens[start:start + bs])
            self.block_pool.cache_full_block(block_id, h)
            parent_hash = h
            n_cached += 1
        return n_cached

    def free_request(self, req):
        """请求结束后释放引用，计数归零的 block 回到空闲队列。"""
        self.block_pool.free_blocks_of(self.block_tables.pop(req.request_id, []))


kv_manager = KVCacheManager(pool)

## 五、亲眼看到前缀命中跳过了多少计算

In [ ]:
PREFIX_LEN, SUFFIX_LEN = 256, 32
prefix_tokens = list(range(10000, 10000 + PREFIX_LEN))


def make_request(rid, suffix_start):
    prompt = prefix_tokens + list(range(suffix_start, suffix_start + SUFFIX_LEN))
    return Request(rid, prompt, 4)


# 请求 A：冷启动，全量 prefill，然后把自己的 block 缓存起来
req_a = make_request("A", 20000)
print(f"请求 A 入场: 命中 0 个 block，需要计算 {req_a.num_tokens_to_schedule()} token")
kv_manager.allocate_slots(req_a, req_a.num_tokens_to_schedule())
req_a.num_computed_tokens = req_a.num_prompt_tokens       # 模拟"算完了"
n_cached = kv_manager.cache_blocks(req_a)
print(f"  A 算完后缓存了 {n_cached} 个 block，缓存表大小 {len(pool.cached_block_hash_to_block)}")
print(f"  A 的 block_table: {kv_manager.block_tables['A']}")

# 请求 B：共享同一段前缀，应该直接命中
req_b = make_request("B", 30000)
hit = kv_manager.attach_prefix_cache(req_b)
print()
print(f"请求 B 入场: 命中 {hit} 个 block")
print(f"  num_computed_tokens 被直接推进到 {req_b.num_computed_tokens}/{req_b.num_prompt_tokens}")
print(f"  还需要计算的 token 数: {req_b.num_tokens_to_schedule()}   ← 只剩后缀")

# 请求 C：前缀完全不同，命中 0
req_c = Request("C", list(range(70000, 70000 + PREFIX_LEN)), 4)
hit_c = kv_manager.attach_prefix_cache(req_c)
print()
print(f"请求 C（前缀完全不同）: 命中 {hit_c} 个 block，仍需计算 {req_c.num_tokens_to_schedule()} token")

print()
print("看 B 和 C 的对比——这就是前缀缓存的全部收益：")
print("  命中的 token 一个都不用算，num_computed_tokens 被直接推上去。")
print("  在 vLLM 里这个字段会被 Scheduler 读走，用来算本轮该给这个请求排多少 token。")

### 引用计数：共享但不复制

In [ ]:
shared = kv_manager.block_tables["B"]
print("请求 A 和 B 共享的 block：")
for bid in shared[:4]:
    print(f"  {pool.blocks[bid]}")
print("  ...")

kv_manager.free_request(req_a)
print()
print("释放请求 A 之后：")
for bid in shared[:4]:
    print(f"  {pool.blocks[bid]}   ← B 还引用着，没有回到空闲队列")

kv_manager.free_request(req_b)
print()
print("释放请求 B 之后：")
for bid in shared[:4]:
    print(f"  {pool.blocks[bid]}   ← 引用计数归零，块回到空闲队列")

print(f"\n空闲 block 数: {pool.num_free}")
print()
print("引用计数是前缀共享能成立的关键：多个请求读同一份物理块，谁都不复制。")
print("一旦某条序列要往里写（分叉后产生新 token），就需要写时复制（COW）——")
print("vLLM 里的做法是 block 被覆写前先从缓存哈希表里摘除。")

## 六、验证：复用前缀算出来的结果必须一致

上面演示的是**账本**。账本对了不代表算得对——复用前缀的 KV 和整体重算，输出必须一致。这个验证不能省，因为复用错了不会报错，只会悄悄让输出变差。

In [ ]:
model = build_model(block_size=4096)
P, S = 512, 128

prefix_ids = torch.randint(0, model.cfg.vocab_size, (1, P), device=DEVICE)
suffix_ids = torch.randint(0, model.cfg.vocab_size, (1, S), device=DEVICE)

logits_full, _ = model(torch.cat([prefix_ids, suffix_ids], dim=1))       # 整体重算
_, prefix_past = model(prefix_ids)
logits_reuse, _ = model(suffix_ids, past_kvs=prefix_past, pos_offset=P)  # 复用前缀 KV

diff = (logits_full[:, -1] - logits_reuse[:, -1]).abs().max().item()
same = torch.equal(logits_full[:, -1].argmax(-1), logits_reuse[:, -1].argmax(-1))

print(f"最后一位 logits 最大差异: {diff:.2e}")
print(f"argmax 结果一致        : {same}")
print()
print("差异应该在 1e-3 量级以下（浮点运算顺序导致的舍入），但 argmax 必须完全一致。")
print("如果不一致，说明位置偏移或掩码算错了——这是复用前缀时最容易出的 bug。")

## 七、block size 怎么选

分页不是免费的：block 末尾用不满的部分就是**内部碎片**。block 越小碎片越少，但 block 表越长、寻址开销越大。

In [ ]:
SEQ_LEN = 8192
print(f"以一条 {SEQ_LEN} token 的序列为例：\n")
print(f"{'block_size':>12}{'平均内部碎片':>18}{'block 表项数':>16}")
print("-" * 48)
for bs in [1, 4, 8, 16, 32, 64, 128, 256]:
    print(f"{bs:>12}{(bs - 1) / 2:>14.1f} token{SEQ_LEN // bs:>15}")

print()
print("怎么权衡：")
print("  block_size 太小 → block 表很长，attention 要遍历更多块，索引开销上升")
print("  block_size 太大 → 内部碎片严重，短请求的显存被浪费")
print("  16 是 vLLM 的默认值：平均碎片 7.5 个 token，8K 序列表长 512，两边都还能接受")
print()
print("面试时能说出'这是碎片和寻址开销的折中，而且 attention kernel 对这个值有约束'，")
print("就明显高出一个层次。")

## 八、分页到底省了多少

In [ ]:
def utilization(strategy, lens, block_size=16, max_len=4096):
    ideal = sum(lens)
    if strategy == "reserve":      # 每条序列预留最大长度
        allocated = len(lens) * max_len
    elif strategy == "padded":     # 第 04 章的填充方案：按 batch 内最长对齐
        allocated = len(lens) * max(lens)
    else:                          # 分页按需
        allocated = sum(math.ceil(L / block_size) * block_size for L in lens)
    return ideal / allocated


workloads = {
    "8 条短请求 (100)": [100] * 8,
    "8 条长请求 (4096)": [4096] * 8,
    "长短混合 (100~4K)": [100, 200, 400, 800, 1600, 3200, 4096, 4096],
    "8 条刚起步 (10)": [10] * 8,
}

print(f"{'场景':<24}{'预留最大长度':>14}{'对填充(第04章)':>16}{'分页按需':>12}")
print("-" * 68)
for name, lens in workloads.items():
    print(f"{name:<24}{utilization('reserve', lens) * 100:>13.1f}%"
          f"{utilization('padded', lens) * 100:>15.1f}%{utilization('paged', lens) * 100:>11.1f}%")

print()
print("中间那列就是第 04 章 ModelRunner 实际在做的事——为了对齐而填充。")
print("请求刚起步、只用了 10 个 token 时，填充方案浪费 99.8%，分页只损失 block 内取整。")

## 九、前缀命中的规律（必背）

In [ ]:
def block_hashes(tokens, bs):
    """整条序列的链式哈希列表。末尾不足一个 block 的 token 不参与。"""
    parent, out = -1, []
    for i in range(0, len(tokens) - bs + 1, bs):
        h = hash_block_tokens(parent, tokens[i:i + bs])
        out.append(h)
        parent = h
    return out


def matched_blocks(a, b, bs):
    """两条序列从开头起连续匹配的 block 数。"""
    n = 0
    for x, y in zip(block_hashes(a, bs), block_hashes(b, bs)):
        if x != y:
            break
        n += 1
    return n


BS = 16
BASE = list(range(1000, 1128))    # 128 token 公共部分 = 8 个 block
base = BASE + [1, 2, 3, 4]

variants = {
    "同前缀，后缀不同": BASE + [9, 8, 7, 6],
    "前缀后追加变量": BASE + [555] + [1, 2, 3],
    "变量插在最开头": [777] + BASE,
    "变量插在第 64 token 后": BASE[:64] + [888] + BASE[64:],
}

print(f"对照组有 {len(block_hashes(base, BS))} 个可缓存 block"
      f"（末尾不足一块的 4 个 token 不算）\n")
print(f"{'变体':<26}{'命中block':>10}{'判定':>16}")
print("-" * 52)
for name, seq in variants.items():
    m = matched_blocks(base, seq, BS)
    verdict = "高" if m >= 6 else ("低" if m > 0 else "完全失效")
    print(f"{name:<26}{m:>10}{verdict:>16}")

**这张表就是 prefix cache 的全部行为规律：**

- 后缀不同不影响命中——这很好，检索到的文档本来就不一样。
- 前缀后追加内容不影响命中——前面的 block 已经完整且固定了。
- **变量插在最开头，命中率直接归零**。哪怕后面 100 个 token 完全相同，第一个 block 变了，链式哈希全断。
- 变量插在中间，只有它之前的 block 能命中。

所以有一条工程铁律：**system prompt、指令模板、few-shot 放最前面，变量放最后面。**

上线前值得做一次审计：把线上 prompt 的每个字段按位置排一排，看哪个字段会让缓存整段失效。这通常是**改一行位置换来 30% 成本下降**的优化。

## 十、API 对照表

| 本章的方法 | vLLM 里的对应物 | 怎么找 |
|---|---|---|
| `KVCacheManager.attach_prefix_cache()` | `get_computed_blocks()` + `allocate_slots()` | `rg "def get_computed_blocks" vllm/v1/core/kv_cache_manager.py` |
| `KVCacheManager.allocate_slots()` | `allocate_slots()` | 同上文件 |
| `KVCacheManager.cache_blocks()` | `cache_full_blocks()` | `rg "def cache_full_blocks" vllm/v1/core/block_pool.py` |
| `KVCacheManager.free_request()` | `free()` | `vllm/v1/core/kv_cache_manager.py` |
| `BlockPool.get_new_blocks()` | `get_new_blocks()` | `vllm/v1/core/block_pool.py` |
| `BlockPool.free_blocks_of()` | `free_blocks()` | 同上 |
| `BlockPool.cached_block_hash_to_block` | 同名字段 | 同上 |
| `KVCacheBlock.ref_cnt / block_hash` | 同名字段 | `vllm/v1/core/kv_cache_utils.py` |
| `hash_block_tokens()` | `hash_block_tokens()` | `vllm/v1/core/kv_cache_utils.py` |
| `BlockPool.free_blocks`（deque） | `FreeKVCacheBlockQueue` | `vllm/v1/core/kv_cache_utils.py` |

**字段名和方法名几乎完全一致，这是故意的。** 你现在打开 `block_pool.py`，看到的应该是一堆熟悉的东西。

## 十一、面试话术

**问：PagedAttention 解决什么问题？** 分三个层次答：

1. **消除外部碎片**：KV 不再要求物理连续，按 block 按需分配。
2. **消除对齐填充**：不同长度的序列能进同一个 batch，不需要补齐。第 04 章的填充方案在请求刚起步时浪费 99.8%，分页只损失 block 内取整。
3. **支持零拷贝前缀共享**：block 按引用计数共享，多条请求读同一份物理块。

**问：前缀缓存命中之后发生了什么？**

答：请求的 `num_computed_tokens` 被直接推进到命中长度，这些 token 一个都不用算。Scheduler 下一轮读这个字段，就知道只该给剩下那点后缀排 token。**命中和不命中，在 vLLM 里体现为同一个字段的不同取值，而不是两套代码路径。**

**问：block size 怎么选？** 内部碎片和寻址开销的折中。小了碎片少但表长、开销大；大了反过来。vLLM 默认 16。

**问：prefix cache 什么情况会完全失效？** **前缀第一个 block 就不同**。最常见原因是 prompt 把变量（用户 query、时间戳、请求 ID）放在最前面。链式哈希必须从头连续匹配，第一个 block 断了后面全断。

**作业**

1. 把 `BS` 改成 32 和 8，重跑命中实验，观察"变量插在中间"那个用例的命中 block 数怎么变。
2. 给 `BlockPool` 加一个 LRU 淘汰：`get_new_blocks` 找不到空闲块时，优先淘汰哈希表中引用计数为 0 的 block。想想这和 vLLM 的 eviction queue 是不是同一个东西。
3. 思考题：两条序列共享了一个 block，其中一条分叉产生了新 token，怎么处理？（提示：写时复制 COW，vLLM 里是覆写前先摘除缓存哈希）

**下一章**：把 `Scheduler` 的 token 预算调小，看 chunked prefill 怎么自然浮现出来。